## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 01 - Prepare Original, Full-PNG, and YOLO ROI Datasets |
| Model / workflow | DenseNet-121/201 |
| Input | 224x224 |
| Loss | not reported |
| Training / pipeline | YOLO detection/ROI workflow |
| Result | Full bilateral HDF5 source: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/DetKneeData/H5 |


# 01 - Prepare Original, Full-PNG, and YOLO ROI Datasets

Run this notebook once. It uses the one existing `KneeXrayData.zip` archive, creates full bilateral PNG images, then produces a resumable square-ROI dataset with `train`, `val`, and `test` splits.

It does not train a classifier. The original 224x224 published images are retained in place; they are not copied or modified.


In [ ]:
!pip -q install "ultralytics>=8.3,<9" "h5py>=3.9"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:0000:01


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import math
import os
import time
from pathlib import Path
from zipfile import ZipFile

import cv2
import h5py
import numpy as np
import torch
from ultralytics import YOLO


Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Extract the source archive only when required

Both source folders are required: the published 224x224 PNGs provide KL labels, while the HDF5 files contain full bilateral radiographs.


In [ ]:
ARCHIVE = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip")
EXTRACT_DIR = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted")
DATA_ROOT = EXTRACT_DIR / "KneeXrayData"
ORIGINAL_ROOT = DATA_ROOT / "ClsKLData/kneeKL224"
H5_ROOT = DATA_ROOT / "DetKneeData/H5"

if not ARCHIVE.is_file():
    raise FileNotFoundError(f"ZIP file not found: {ARCHIVE}")
if not ORIGINAL_ROOT.is_dir() or not H5_ROOT.is_dir():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ARCHIVE) as archive:
        archive.extractall(EXTRACT_DIR)
for required in (ORIGINAL_ROOT, H5_ROOT):
    if not required.is_dir():
        raise FileNotFoundError(f"Required source folder not found: {required}")

print("Original 224x224 dataset:", ORIGINAL_ROOT)
print("Full bilateral HDF5 source:", H5_ROOT)


Original 224x224 dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
Full bilateral HDF5 source: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/DetKneeData/H5


## Export full bilateral radiographs as PNG

Each bilateral image is saved once by patient ID. It cannot be placed in a single KL-grade folder because its left and right knees can have different grades. This cell resumes safely by skipping PNGs already written.


In [ ]:
FULL_PNG_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2"
)
SPLITS = ("train", "val", "test")

for split in SPLITS:
    source_dir = H5_ROOT / f"{split}H5"
    output_dir = FULL_PNG_ROOT / split
    if not source_dir.is_dir():
        raise FileNotFoundError(source_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    h5_paths = sorted(source_dir.glob("*.h5"))
    for index, h5_path in enumerate(h5_paths, start=1):
        output_path = output_dir / f"{h5_path.stem}.png"
        if output_path.is_file():
            continue
        with h5py.File(h5_path, "r") as handle:
            image = np.asarray(handle["images"])
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2)
        if image.shape[-1] == 1:
            image = np.repeat(image, 3, axis=2)
        image = np.clip(image[..., :3], 0, 255).astype(np.uint8)
        if not cv2.imwrite(str(output_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR)):
            raise RuntimeError(f"Cannot write {output_path}")
        if index % 100 == 0 or index == len(h5_paths):
            print(f"{split}: {index}/{len(h5_paths)} full PNG images")


train: 100/2889 full PNG images
train: 200/2889 full PNG images
train: 300/2889 full PNG images
train: 400/2889 full PNG images
train: 500/2889 full PNG images
train: 600/2889 full PNG images
train: 700/2889 full PNG images
train: 800/2889 full PNG images
train: 900/2889 full PNG images
train: 1000/2889 full PNG images
train: 1100/2889 full PNG images
train: 1200/2889 full PNG images
train: 1300/2889 full PNG images
train: 1400/2889 full PNG images
train: 1500/2889 full PNG images
train: 1600/2889 full PNG images
train: 1700/2889 full PNG images
train: 1800/2889 full PNG images
train: 1900/2889 full PNG images
train: 2000/2889 full PNG images
train: 2100/2889 full PNG images
train: 2200/2889 full PNG images
train: 2300/2889 full PNG images
train: 2400/2889 full PNG images
train: 2500/2889 full PNG images
train: 2600/2889 full PNG images
train: 2700/2889 full PNG images
train: 2800/2889 full PNG images
train: 2889/2889 full PNG images
val: 100/413 full PNG images
val: 200/413 full PNG i

## Create the square YOLO ROI dataset

The source boxes are expanded by 1.15x, then the shorter dimension is extended to form a square. Padding is used only where that square would cross the source-image boundary. No center crop is used. Existing ROIs are skipped, so this cell resumes after a Colab interruption.


In [ ]:
YOLO_CHECKPOINT = Path("/content/drive/MyDrive/Models/yolov8_checkpoint/best.pt")
ROI_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "densenet121_yolo_square_roi_trainvaltest_v2"
)
YOLO_BATCH_SIZE = 32
YOLO_CONFIDENCE = 0.45
YOLO_IMAGE_SIZE = 640
BOX_EXPANSION = 1.15

if not YOLO_CHECKPOINT.is_file():
    raise FileNotFoundError(YOLO_CHECKPOINT)

labels = {}
for split in SPLITS:
    for grade in range(5):
        for path in (ORIGINAL_ROOT / split / str(grade)).glob("*.png"):
            labels[(split, path.stem[:-1], path.stem[-1].upper())] = grade


def make_square_roi(image, box):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    box_width, box_height = x2 - x1, y2 - y1
    if box_width <= 0 or box_height <= 0:
        raise ValueError(f"Invalid YOLO box: {box}")
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    side = int(math.ceil(max(box_width, box_height) * BOX_EXPANSION))
    wanted_x1 = int(math.floor(center_x - side / 2))
    wanted_y1 = int(math.floor(center_y - side / 2))
    wanted_x2, wanted_y2 = wanted_x1 + side, wanted_y1 + side
    crop = image[max(0, wanted_y1):min(height, wanted_y2), max(0, wanted_x1):min(width, wanted_x2)]
    if crop.size == 0:
        raise RuntimeError(f"Empty ROI crop for {box}")
    return cv2.copyMakeBorder(
        crop,
        max(0, -wanted_y1), max(0, wanted_y2 - height),
        max(0, -wanted_x1), max(0, wanted_x2 - width),
        cv2.BORDER_CONSTANT, value=(0, 0, 0),
    )


detector = YOLO(str(YOLO_CHECKPOINT))
detector_device = 0 if torch.cuda.is_available() else "cpu"
for split in SPLITS:
    for grade in range(5):
        (ROI_ROOT / split / str(grade)).mkdir(parents=True, exist_ok=True)
    image_paths = sorted((FULL_PNG_ROOT / split).glob("*.png"))
    pending = []
    for image_path in image_paths:
        patient = image_path.stem
        required = [ROI_ROOT / split / str(labels[(split, patient, side)]) / f"{patient}{side}.png" for side in ("R", "L")]
        if not all(path.is_file() for path in required):
            pending.append(image_path)

    for start in range(0, len(pending), YOLO_BATCH_SIZE):
        batch_paths = pending[start:start + YOLO_BATCH_SIZE]
        batch_images = [cv2.imread(str(path), cv2.IMREAD_COLOR) for path in batch_paths]
        if any(image is None for image in batch_images):
            bad = batch_paths[next(i for i, image in enumerate(batch_images) if image is None)]
            raise RuntimeError(f"Cannot read full PNG: {bad}")
        predictions = detector.predict(
            source=batch_images, conf=YOLO_CONFIDENCE, imgsz=YOLO_IMAGE_SIZE,
            device=detector_device, batch=YOLO_BATCH_SIZE, save=False, verbose=False,
        )
        for image_path, image, prediction in zip(batch_paths, batch_images, predictions):
            patient = image_path.stem
            boxes = prediction.boxes.xyxy.detach().cpu().numpy()
            scores = prediction.boxes.conf.detach().cpu().numpy()
            boxes = boxes[np.argsort(scores)[::-1][:2]]
            boxes = sorted(boxes, key=lambda box: float(box[0] + box[2]))
            if len(boxes) != 2:
                raise RuntimeError(f"Expected two knee detections: {image_path}")
            for box, side in zip(boxes, ("R", "L")):
                grade = labels.get((split, patient, side))
                if grade is None:
                    raise RuntimeError(f"Missing label: {split}/{patient}{side}")
                destination = ROI_ROOT / split / str(grade) / f"{patient}{side}.png"
                if not destination.is_file() and not cv2.imwrite(str(destination), make_square_roi(image, box)):
                    raise RuntimeError(f"Cannot write {destination}")
        print(f"{split}: {min(start + len(batch_paths), len(pending))}/{len(pending)} pending bilateral images")

print("ROI dataset ready:", ROI_ROOT)


train: 32/2889 pending bilateral images
train: 64/2889 pending bilateral images
train: 96/2889 pending bilateral images
train: 128/2889 pending bilateral images
train: 160/2889 pending bilateral images
train: 192/2889 pending bilateral images
train: 224/2889 pending bilateral images
train: 256/2889 pending bilateral images
train: 288/2889 pending bilateral images
train: 320/2889 pending bilateral images
train: 352/2889 pending bilateral images
train: 384/2889 pending bilateral images
train: 416/2889 pending bilateral images
train: 448/2889 pending bilateral images
train: 480/2889 pending bilateral images
train: 512/2889 pending bilateral images
train: 544/2889 pending bilateral images
train: 576/2889 pending bilateral images
train: 608/2889 pending bilateral images
train: 640/2889 pending bilateral images
train: 672/2889 pending bilateral images
train: 704/2889 pending bilateral images
train: 736/2889 pending bilateral images
train: 768/2889 pending bilateral images
train: 800/2889 pen

## Confirm Drive writes before optional shutdown

This is a practical mounted-Drive check: it waits until file count and total size are stable for 60 seconds, calls `os.sync()`, then optionally powers off the Colab VM. Set `AUTO_SHUTDOWN` to `True` only in the last run of this notebook.


In [ ]:
WATCHED_DIRS = [FULL_PNG_ROOT, ROI_ROOT]
POLL_SECONDS = 10
STABLE_SECONDS = 60
AUTO_SHUTDOWN = False


def snapshot(path):
    files = [file_path for file_path in path.rglob("*") if file_path.is_file()]
    return len(files), sum(file_path.stat().st_size for file_path in files)


previous = None
stable_since = None
while True:
    current = {str(path): snapshot(path) for path in WATCHED_DIRS}
    print(current)
    if current == previous:
        stable_since = stable_since or time.time()
        if time.time() - stable_since >= STABLE_SECONDS:
            break
    else:
        previous, stable_since = current, None
    time.sleep(POLL_SECONDS)

os.sync()
print("Drive outputs are stable and sync has been requested.")
if AUTO_SHUTDOWN:
    print("Shutting down Colab in 15 seconds...")
    time.sleep(15)
    os.system("sudo shutdown -h now")


{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/full_bilateral_png_v2': (4130, 514256090), '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2': (8260, 123954038)}
{'/content/drive/MyDrive/Datasets/Kn